# Segmentação de Água em Imagens de Satélite — Pipeline Final

**Competição:** 2026 ITU · Ingenuity Cup AI and Space Computing Challenge  
**Dataset:** GID (Gaofen Image Dataset)  
**Autor:** Luiz Felipe Castro — luiz.castro@usp.br  

---

Pipeline completo de treinamento e geração de submissão. Para entender a progressão de tentativas que levou a estas escolhas, consulte `01_exploracao_tentativas.ipynb`.

## Decisões de arquitetura

**Modelo:** UNet++ com encoder EfficientNet-b1 e atenção scSE  
O UNet++ usa conexões densas entre os blocos do encoder e decoder, o que melhora a captura de estruturas em múltiplas escalas — importante para detectar tanto lagos grandes quanto rios finos. A atenção scSE recalibra as features tanto espacialmente quanto por canal, ajudando o modelo a focar nas regiões com água.

O EfficientNet-b1 foi escolhido sobre o b3 por limitação de memória da GPU P100 do Kaggle (16GB): o b3 causava OOM com batch size 2 e acumulação de gradiente, enquanto o b1 rodava de forma estável.

**Loss:** FocalTversky + BCE ponderada + Lovász  
Combinação voltada para dados com forte desbalanceamento de classes. A FocalTversky penaliza mais os falsos negativos (água não detectada). A BCE com pos_weight estimado dinamicamente compensa o ratio real entre classes. A Lovász aproxima o IoU diretamente.

**Validação:** split estratificado por presença de água (80/20)  
Garante que ambos os conjuntos tenham representação proporcional de imagens com e sem água, evitando que a validação fique enviesada para um dos casos.


## 1. Imports e configuração

In [ ]:
import os, glob, time, zipfile, cv2, gc, torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import segmentation_models_pytorch as smp

START_TIME = time.time()
MAX_RUNTIME_SECONDS = 8 * 60 * 60  # limite de 8h para a sessão do Kaggle
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE    = (512, 512)
BATCH_SIZE  = 2   # limitado pela VRAM da P100
ACCUM_STEPS = 8   # simula batch efetivo de 16
TOTAL_EPOCHS = 100
THRESHOLD    = 0.35  # limiar de decisão ajustado visualmente
RANDOM_STATE = 42

CHECKPOINT_SAVE_PATH = '/kaggle/working/last_checkpoint.pth'
BEST_MODEL_PATH      = '/kaggle/working/best_water_model.pth'
SUBMISSION_DIR       = '/kaggle/working/submission_masks/'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

BASE_PATH   = '/kaggle/input/datasets/castroluiz/data-zero2x/data'
IMG_FOLDERS = [f'{BASE_PATH}/Train/GID-img-{i}' for i in range(1, 5)]
LBL_DIR     = f'{BASE_PATH}/Train/GID-label'
TEST_DIR    = f'{BASE_PATH}/Test/Images'

print(f'Device: {DEVICE}')
print(f'Batch efetivo: {BATCH_SIZE * ACCUM_STEPS}')


## 2. Dataset

In [ ]:
class WaterDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs     = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_p, lbl_p = self.pairs[idx]
        image    = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        mask_raw = cv2.imread(lbl_p, cv2.IMREAD_GRAYSCALE)
        # valor 29 corresponde à classe água no GID — confirmado por inspeção visual
        mask = (mask_raw == 29).astype(np.float32)
        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask'].unsqueeze(0)
        return image, mask


# Augmentations de treino mais agressivas para rios finos
# RandomResizedCrop com scale baixo (0.3) força o modelo a aprender
# a detectar água em recortes onde ela pode ocupar grande parte do frame
train_transform = A.Compose([
    A.RandomResizedCrop(
        size=IMG_SIZE, scale=(0.3, 1.0), ratio=(0.75, 1.33),
        interpolation=cv2.INTER_LANCZOS4, p=1.0
    ),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.OneOf([
        A.ElasticTransform(alpha=120, sigma=6, p=1.0),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
        A.OpticalDistortion(distort_limit=0.5, p=1.0),
    ], p=0.3),
    A.OneOf([
        A.RandomBrightnessContrast(0.3, 0.3, p=1.0),
        A.HueSaturationValue(20, 30, 20, p=1.0),
        A.CLAHE(clip_limit=4.0, p=1.0),
    ], p=0.5),
    A.OneOf([
        A.GaussNoise(std_range=(10.0/255, 50.0/255), p=1.0),
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.3),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(8, 32), hole_width_range=(8, 32), p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1], interpolation=cv2.INTER_LANCZOS4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


In [ ]:
# Monta os pares (imagem, máscara) varrendo as pastas de treino
all_pairs = []
label_files = {
    os.path.splitext(os.path.basename(f))[0]: f
    for f in glob.glob(os.path.join(LBL_DIR, '*.png'))
}
for folder in IMG_FOLDERS:
    for img_path in glob.glob(os.path.join(folder, '*.jpg')):
        stem = os.path.splitext(os.path.basename(img_path))[0]
        if stem in label_files:
            all_pairs.append((img_path, label_files[stem]))

print(f'Total de pares encontrados: {len(all_pairs)}')

# Split estratificado por presença de água
# Garante que treino e validação tenham proporção similar de imagens com água
water_pairs    = []
no_water_pairs = []
for pair in tqdm(all_pairs, desc='Classificando imagens'):
    m = cv2.imread(pair[1], cv2.IMREAD_GRAYSCALE)
    if (m == 29).any():
        water_pairs.append(pair)
    else:
        no_water_pairs.append(pair)

print(f'Com água: {len(water_pairs)} | Sem água: {len(no_water_pairs)}')

val_w  = max(1, int(0.2 * len(water_pairs)))
val_nw = max(1, int(0.2 * len(no_water_pairs)))
train_pairs = water_pairs[val_w:] + no_water_pairs[val_nw:]
val_pairs   = water_pairs[:val_w] + no_water_pairs[:val_nw]

train_loader = DataLoader(WaterDataset(train_pairs, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(WaterDataset(val_pairs, val_transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Treino: {len(train_pairs)} | Validação: {len(val_pairs)}')


## 3. Modelo

In [ ]:
model = smp.UnetPlusPlus(
    encoder_name='efficientnet-b1',   # b3 causava OOM na P100 com batch 2
    encoder_weights='imagenet',
    decoder_attention_type='scse',    # atenção espacial + de canal
    in_channels=3,
    classes=1,
    activation=None,                  # sigmoid aplicado manualmente para compatibilidade com as losses
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parâmetros totais: {total_params:,}')


## 4. Função de perda

In [ ]:
class TverskyLoss(nn.Module):
    """Generalização da Dice Loss. alpha > beta penaliza mais falsos negativos,
    útil quando perder água não detectada é pior do que detectar água onde não há."""
    def __init__(self, alpha=0.7, beta=0.3, smooth=1e-6):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.smooth = smooth

    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits)
        p_flat  = probs.view(-1)
        t_flat  = targets.view(-1)
        TP = (p_flat * t_flat).sum()
        FP = (p_flat * (1 - t_flat)).sum()
        FN = ((1 - p_flat) * t_flat).sum()
        tversky = (TP + self.smooth) / (TP + self.alpha * FN + self.beta * FP + self.smooth)
        return 1 - tversky


class FocalTverskyLoss(nn.Module):
    """Aplica um expoente à Tversky Loss para aumentar o peso nos exemplos difíceis."""
    def __init__(self, alpha=0.7, beta=0.3, gamma=1.5, smooth=1e-6):
        super().__init__()
        self.tversky = TverskyLoss(alpha, beta, smooth)
        self.gamma   = gamma

    def forward(self, logits, targets):
        return self.tversky(logits, targets) ** self.gamma


def estimate_pos_weight(pairs, sample_n=50):
    """Estima o desbalanceamento real do dataset para calibrar a BCE."""
    sampled = pairs[:sample_n]
    total_pos, total_neg = 0, 0
    for _, lbl_p in sampled:
        m   = cv2.imread(lbl_p, cv2.IMREAD_GRAYSCALE)
        pos = (m == 29).sum()
        total_pos += pos
        total_neg += m.size - pos
    ratio = total_neg / (total_pos + 1e-7)
    print(f'pos_weight estimado: {ratio:.1f}x (limitado a 30)')
    return min(ratio, 30.0)


class CombinedLoss(nn.Module):
    """Combinação de três losses complementares para dados com forte desbalanceamento.
    FocalTversky (60%) foca nos falsos negativos e exemplos difíceis.
    BCE ponderada (20%) penaliza erros pixel a pixel com peso proporcional ao desbalanceamento.
    Lovász (20%) aproxima o IoU diretamente, melhorando a qualidade das bordas.
    """
    def __init__(self, pos_weight_value=20.0):
        super().__init__()
        self.focal_tversky = FocalTverskyLoss(alpha=0.7, beta=0.3, gamma=1.5)
        self.bce    = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value]).to(DEVICE))
        self.lovasz = smp.losses.LovaszLoss(mode='binary', per_image=False)

    def forward(self, logits, targets):
        return (
            0.6 * self.focal_tversky(logits, targets) +
            0.2 * self.bce(logits, targets) +
            0.2 * self.lovasz(logits, targets)
        )


pos_weight_val = estimate_pos_weight(train_pairs)
criterion      = CombinedLoss(pos_weight_value=pos_weight_val)


## 5. Otimizador e scheduler

In [ ]:
# Learning rates diferentes para encoder e decoder
# O encoder já vem pré-treinado no ImageNet, então recebe lr menor
# para não destruir os pesos aprendidos
encoder_params = list(model.encoder.parameters())
decoder_params = [p for n, p in model.named_parameters() if not n.startswith('encoder')]

optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': 1e-4, 'weight_decay': 1e-4},
    {'params': decoder_params, 'lr': 5e-4, 'weight_decay': 1e-4},
])

# CosineAnnealingWarmRestarts reinicia o lr periodicamente
# ajuda a escapar de mínimos locais durante o treinamento longo
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-6
)

scaler = GradScaler('cuda')  # precisão mista para economizar VRAM


## 6. Métricas

In [ ]:
def calculate_kappa(tp, fp, fn, tn):
    """Kappa de Cohen — métrica oficial da competição.
    Mede a concordância entre predição e ground truth descontando o acerto por acaso.
    Valor 1.0 = perfeito, 0.0 = equivalente a chutar aleatoriamente.
    """
    total = tp + fp + fn + tn + 1e-7
    po    = (tp + tn) / total
    pe    = ((tp + fp) * (tp + fn) + (tn + fp) * (tn + fn)) / (total ** 2 + 1e-7)
    return (po - pe) / (1 - pe + 1e-7)


def calculate_metrics(preds_bin, targets):
    p = preds_bin.view(-1).long()
    t = targets.view(-1).long()
    tp = ((p == 1) & (t == 1)).sum().item()
    fp = ((p == 1) & (t == 0)).sum().item()
    fn = ((p == 0) & (t == 1)).sum().item()
    tn = ((p == 0) & (t == 0)).sum().item()
    return tp, fp, fn, tn


## 7. Checkpoint

In [ ]:
start_epoch = 0
best_kappa  = -1.0

# Procura checkpoint em inputs do Kaggle (sessão anterior) ou em working (mesma sessão)
checkpoints_input = glob.glob('/kaggle/input/**/last_checkpoint.pth', recursive=True)
if checkpoints_input:
    load_path = checkpoints_input[0]
elif os.path.exists(CHECKPOINT_SAVE_PATH):
    load_path = CHECKPOINT_SAVE_PATH
else:
    load_path = None

if load_path:
    print(f'Retomando de: {load_path}')
    ckpt = torch.load(load_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_kappa  = ckpt['best_kappa']
    print(f'Época: {start_epoch} | Melhor Kappa anterior: {best_kappa:.4f}')
else:
    print('Nenhum checkpoint encontrado. Iniciando do zero.')


## 8. Treinamento

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, accum_steps, desc):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, (images, masks) in enumerate(tqdm(loader, desc=desc, leave=True)):
        images = images.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE, non_blocking=True)
        with autocast('cuda'):
            loss = criterion(model(images), masks) / accum_steps
        scaler.scale(loss).backward()
        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item() * accum_steps
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, threshold=THRESHOLD):
    model.eval()
    total_loss = 0.0
    tp_t = fp_t = fn_t = tn_t = 0
    for images, masks in tqdm(loader, desc='Val', leave=True):
        images = images.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE, non_blocking=True)
        with autocast('cuda'):
            logits = model(images)
            total_loss += criterion(logits, masks).item()
        preds_bin = (torch.sigmoid(logits) > threshold).float()
        tp, fp, fn, tn = calculate_metrics(preds_bin, masks)
        tp_t += tp; fp_t += fp; fn_t += fn; tn_t += tn
    iou    = tp_t / (tp_t + fp_t + fn_t + 1e-7)
    recall = tp_t / (tp_t + fn_t + 1e-7)
    kappa  = calculate_kappa(tp_t, fp_t, fn_t, tn_t)
    return total_loss / len(loader), iou, recall, kappa


In [ ]:
print(f'Épocas: {start_epoch} → {TOTAL_EPOCHS} | Device: {DEVICE}\n')

for epoch in range(start_epoch, TOTAL_EPOCHS):

    # Interrompe se sobrar menos de 10 minutos na sessão do Kaggle
    if (time.time() - START_TIME) > MAX_RUNTIME_SECONDS - 600:
        print('Limite de tempo atingido. Salvando checkpoint...')
        break

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, ACCUM_STEPS,
        desc=f'Epoch [{epoch+1}/{TOTAL_EPOCHS}]'
    )
    val_loss, val_iou, val_recall, val_kappa = validate(model, val_loader, criterion)
    scheduler.step(epoch + 1)

    print(f'  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} '
          f'| IoU: {val_iou:.4f} | Recall: {val_recall:.4f} | Kappa: {val_kappa:.4f}')

    if val_kappa > best_kappa:
        best_kappa = val_kappa
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'  Novo melhor Kappa: {best_kappa:.4f}')

    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_kappa':           best_kappa,
    }, CHECKPOINT_SAVE_PATH)

    gc.collect()
    torch.cuda.empty_cache()

print(f'\nTreinamento concluído | Melhor Kappa: {best_kappa:.4f}')


## 9. Submissão com TTA

Test Time Augmentation (TTA): a predição final é a média das probabilidades obtidas com a imagem original e três versões espelhadas. Isso reduz a variância do modelo sem nenhum custo de treinamento adicional.


In [ ]:
def tta_predict(model, image_tensor, threshold=THRESHOLD):
    """Média de 4 predições: original + flip H + flip V + flip HV."""
    model.eval()
    preds = []
    with torch.no_grad(), autocast('cuda'):
        preds.append(torch.sigmoid(model(image_tensor)))
        fh = torch.flip(image_tensor, dims=[3])
        preds.append(torch.flip(torch.sigmoid(model(fh)), dims=[3]))
        fv = torch.flip(image_tensor, dims=[2])
        preds.append(torch.flip(torch.sigmoid(model(fv)), dims=[2]))
        fhv = torch.flip(image_tensor, dims=[2, 3])
        preds.append(torch.flip(torch.sigmoid(model(fhv)), dims=[2, 3]))
    mean_pred = torch.stack(preds).mean(dim=0)
    return (mean_pred > threshold).squeeze().cpu().numpy().astype(np.uint8)


def generate_submission(team_info):
    best_input = glob.glob('/kaggle/input/**/best_water_model.pth', recursive=True)
    load_path  = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else (best_input[0] if best_input else None)

    if load_path:
        model.load_state_dict(torch.load(load_path, map_location=DEVICE))
        print(f'Melhor modelo carregado: {load_path}')
    else:
        print('best_model não encontrado. Usando pesos do último epoch.')

    test_images = sorted(
        glob.glob(os.path.join(TEST_DIR, '*.jpg')) +
        glob.glob(os.path.join(TEST_DIR, '*.png'))
    )
    print(f'Imagens de teste: {len(test_images)}')

    for img_path in tqdm(test_images, desc='Gerando máscaras'):
        image  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        tensor = val_transform(image=image)['image'].unsqueeze(0).to(DEVICE)
        mask   = tta_predict(model, tensor)

        stem     = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(SUBMISSION_DIR, f'{stem}.png')
        cv2.imwrite(out_path, mask)

    zip_path = f'/kaggle/working/{team_info}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in glob.glob(os.path.join(SUBMISSION_DIR, '*.png')):
            zf.write(f, arcname=os.path.basename(f))

    print(f'ZIP gerado: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
    return zip_path


TEAM_INFO = 'EpicOfSun+LuizFelipeCastro+luiz.castro@usp.br'
generate_submission(TEAM_INFO)
